# 03주 · 제어문·함수·반복과 갱신

수치해석및실습 CE273

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 프로그래밍 (3) / 프로그래밍 (4)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week03/CE273_W03.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week03)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 계산식 대조


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)


<a id="s01"></a>

## 별칭·복사·배열 연산

**먼저 예측**: 대입은 복사인가, 같은 객체를 가리키는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
a=[1,2,3];alias=a;copy=a.copy();alias[0]=99
print(a,alias,copy)
assert a[0]==99 and copy[0]==1
v=np.array([1,2,3]);print("리스트*2:",[1,2,3]*2,"배열*2:",v*2)
assert np.array_equal(v*2,[2,4,6])

**조건 변경**: 중첩 리스트에서 얕은 복사로 충분한지 별도 검토

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 입력의 자료형·shape·변경 허용 여부·기대값 명시

**결과 검토**: 리스트 반복과 NumPy 원소별 연산을 구분

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s02"></a>

## 반복 갱신·함수의 반환

**먼저 예측**: 반복 전 값과 갱신 후 값 중 무엇을 다음 단계에 쓰는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
def update(x):
    return 1.5*x+1
x=0.;history=[x]
for k in range(10):
    x=update(x);history.append(x)
print(pd.DataFrame({"k":range(11),"x":history}))
assert np.allclose(history,2*(1.5**np.arange(11)-1))
def show(x):
    print(x)
assert show(3) is None
print("출력과 반환은 별개")

**조건 변경**: 초기값만 0→1로 변경. 초기값·반복 수·갱신식 분리

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: x(k+1)=1.5x(k)+1·x0=0·10회·모든 중간값 반환·검산식 요청

**결과 검토**: 고정 반복과 수렴 반복의 차이. 이 예제는 양의 초기값에서 발산

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

반복·갱신·반환과 검산식 기록

입력 항목:

- `원본_x3`: 해당 계산값·항 수·오차 기준의 수치
- `print_함수는_표시값을_자동반환`: 판단 · Python `True`/`False`
- `갱신계수_1_5의_반복이_항상_수렴`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인


In [ ]:
answers = {'원본_x3': None, 'print_함수는_표시값을_자동반환': None, '갱신계수_1_5의_반복이_항상_수렴': None}


In [ ]:
def check_answers(values):
    expected = {'원본_x3': 4.75, 'print_함수는_표시값을_자동반환': False, '갱신계수_1_5의_반복이_항상_수렴': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
